# TP4 — Reproduction GPU du U-Net BUSI

Notebook de reproduction dérivé du TP4 SUP'COM. Il conserve l'architecture, la construction des masques, `normalize(axis=1)`, le split, les configurations 10/50 époques et les noms de checkpoints. Aucune cellule n'est pré-exécutée et aucun résultat n'est revendiqué.

In [ ]:
from pathlib import Path
import glob
import os
import cv2
import numpy as np
from tensorflow.keras.utils import normalize
from sklearn.model_selection import train_test_split
from core.unet_tp4 import simple_unet_model

print('GPU devices:')
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

Configurez le vrai dossier `Dataset_BUSI_with_GT` et un dossier de sortie persistant. Le TP original utilisait `/content/drive/MyDrive/breast_dataset (1)/Dataset_BUSI_with_GT` et sauvegardait dans `/content/drive/MyDrive/`.

In [ ]:
BASE_DIR = Path(os.environ.get('BUSI_DATASET_DIR', '/content/drive/MyDrive/breast_dataset (1)/Dataset_BUSI_with_GT'))
OUTPUT_DIR = Path(os.environ.get('TP4_OUTPUT_DIR', '/content/drive/MyDrive'))
if not BASE_DIR.is_dir():
    raise FileNotFoundError(f'Dataset BUSI absent: {BASE_DIR}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
SIZE = 256
INCLUDE_NORMAL = False
classes = ['benign', 'malignant'] + (['normal'] if INCLUDE_NORMAL else [])
image_dataset, mask_dataset = [], []
skipped = 0

for cls in classes:
    folder = BASE_DIR / cls
    for name in sorted(os.listdir(folder)):
        if '_mask' in name or not name.lower().endswith('.png'):
            continue
        img = cv2.imread(str(folder / name), 0)
        if img is None:
            skipped += 1
            continue
        img = cv2.resize(img, (SIZE, SIZE))  # TP4: interpolation OpenCV par défaut
        stem = Path(name).stem
        msk = np.zeros((SIZE, SIZE), dtype=np.uint8)
        for mask_file in glob.glob(str(folder / (stem + '_mask*.png'))):
            m = cv2.imread(mask_file, 0)
            if m is not None:
                m = cv2.resize(m, (SIZE, SIZE), interpolation=cv2.INTER_NEAREST)
                msk = np.maximum(msk, m)
        image_dataset.append(img)
        mask_dataset.append(msk)

print('Images:', len(image_dataset), 'Masks:', len(mask_dataset), 'Skipped:', skipped)

In [ ]:
image_dataset = np.array(image_dataset)
image_dataset = normalize(image_dataset, axis=1)
image_dataset = np.expand_dims(image_dataset, axis=3)
mask_dataset = np.array(mask_dataset)
mask_dataset = np.expand_dims(mask_dataset, axis=3)
mask_dataset = (mask_dataset > 127).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    image_dataset, mask_dataset, test_size=0.10, random_state=42
)
print('Train:', X_train.shape, y_train.shape)
print('Val:', X_test.shape, y_test.shape)

## Exécution TP4 — 10 époques, batch 16, `shuffle=False`

In [ ]:
model = simple_unet_model(256, 256, 1)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
batch_size = 16
epochs = 10
history = model.fit(
    X_train, y_train, batch_size=batch_size, verbose=1, epochs=epochs,
    validation_data=(X_test, y_test), shuffle=False
)
model_name = f'unet_breast_batch{batch_size}_epochs{epochs}.hdf5'
model.save(str(OUTPUT_DIR / model_name))
print('Saved:', model_name)

## Exécution TP4 — 50 époques, batch 16
Le TP original n'indiquait pas `shuffle` dans cet appel; le défaut Keras est donc conservé.

In [ ]:
model50 = simple_unet_model(256, 256, 1)
model50.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
batch_size = 16
epochs = 50
history50 = model50.fit(
    X_train, y_train, batch_size=batch_size, verbose=1, epochs=epochs,
    validation_data=(X_test, y_test)
)
model_name = f'unet_breast_batch{batch_size}_epochs{epochs}.hdf5'
model50.save(str(OUTPUT_DIR / model_name))
print('Saved:', model_name)

In [ ]:
def dice_score(a, b):
    a = a.astype(np.uint8).flatten()
    b = b.astype(np.uint8).flatten()
    if a.sum() + b.sum() == 0:
        return 1.0
    return 2.0 * np.sum(a * b) / (a.sum() + b.sum())

def evaluate_like_tp(model_to_evaluate):
    loss, accuracy = model_to_evaluate.evaluate(X_test, y_test, verbose=1)
    prediction = model_to_evaluate.predict(X_test)
    prediction_binary = (prediction > 0.5).astype(np.uint8)
    truth = y_test.astype(np.uint8)
    intersection = np.sum(truth * prediction_binary)
    union = np.sum(truth) + np.sum(prediction_binary) - intersection
    iou = intersection / union
    dices = [dice_score(truth[i], prediction_binary[i]) for i in range(len(X_test))]
    return {'loss': float(loss), 'accuracy': float(accuracy), 'IoU': float(iou), 'mean_dice': float(np.mean(dices))}

print('10 epochs:', evaluate_like_tp(model))
print('50 epochs:', evaluate_like_tp(model50))